# 02 — Data Pipeline

Tokenize & save to **Google Drive** as binary shards.

> Runs on **Google Colab** (free T4 tier).

## 1. Install dependencies

In [ ]:
!pip install -q tokenizers datasets tqdm numpy

## 2. Mount Google Drive (MANDATORY)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration

In [ ]:
import os, math, time, random
import numpy as np
from tqdm.auto import tqdm
from tokenizers import Tokenizer
from datasets import load_dataset

# ─── Tokenizer (Google Drive) ───
TOKENIZER_PATH = "/content/drive/MyDrive/ANTHOR/tokenizer/tokenizer.json"

# ─── Data config (Multiple Sources) ───
DATASETS = [
    {
        "name": "HuggingFaceFW/fineweb-edu",
        "config": "sample-10BT",
        "split": "train",
        "max_docs": 75000,
        "text_key": "text"
    },
    {
        "name": "uonnguyen/vietnamese-wikipedia",
        "config": None,
        "split": "train",
        "max_docs": 75000,
        "text_key": "text"
    }
]

# ─── Sharding ───
TOKENS_PER_SHARD = 100_000_000  # 100M tokens per shard (~200MB)
VAL_SPLIT        = 0.005  # Increased validation split to get more validation tokens

# ─── Google Drive paths (MANDATORY) ───
DRIVE_BASE       = "/content/drive/MyDrive/ANTHOR"
SHARD_DIR        = f"{DRIVE_BASE}/data_shards"
VAL_SHARD_DIR    = f"{DRIVE_BASE}/data_shards_val"
META_PATH        = f"{DRIVE_BASE}/dataset_meta.json"

os.makedirs(SHARD_DIR, exist_ok=True)
os.makedirs(VAL_SHARD_DIR, exist_ok=True)

## 4. Load tokenizer from Drive

In [ ]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
vocab_size = tokenizer.get_vocab_size()
print(f"Loaded tokenizer from Drive: vocab_size={vocab_size}")

## 5. Helper functions

In [ ]:
def write_shard(tokens, shard_dir, shard_idx):
    """Save token array as uint16 .bin shard."""
    arr = np.array(tokens, dtype=np.uint16)
    path = os.path.join(shard_dir, f"shard_{shard_idx:05d}.bin")
    arr.tofile(path)
    return path, len(arr)

def build_dataset(docs, tokenizer, tokens_per_shard, shard_dir, desc="Processing"):
    buffer = []
    shard_idx = 0
    total_tokens = 0
    total_chars = 0
    shard_sizes = []
    
    pbar = tqdm(docs, desc=desc)
    for doc in pbar:
        enc = tokenizer.encode(doc)
        buffer.extend(enc.ids)
        total_chars += len(doc)
        
        while len(buffer) >= tokens_per_shard:
            chunk = buffer[:tokens_per_shard]
            buffer = buffer[tokens_per_shard:]
            _, size = write_shard(chunk, shard_dir, shard_idx)
            total_tokens += size
            shard_sizes.append(size)
            shard_idx += 1
            pbar.set_postfix({"shards": shard_idx, "tokens": f"{total_tokens/1e6:.1f}M"})
    
    if buffer:
        _, size = write_shard(buffer, shard_dir, shard_idx)
        total_tokens += size
        shard_sizes.append(size)
        shard_idx += 1
    
    return {
        "shards": shard_idx,
        "total_tokens": total_tokens,
        "total_chars": total_chars,
        "shard_sizes": shard_sizes,
        "chars_per_token": total_chars / max(total_tokens, 1),
    }

## 6. Load dataset

In [ ]:
docs = []
for ds_info in DATASETS:
    name = ds_info["name"]
    cfg = ds_info.get("config")
    split = ds_info.get("split", "train")
    max_docs = ds_info.get("max_docs", 50000)
    key = ds_info.get("text_key", "text")
    print(f"Loading {name} ({cfg or 'default'}/{split})...")
    try:
        ds = load_dataset(name, cfg, split=split, streaming=True)
        count = 0
        for example in ds:
            if count >= max_docs:
                break
            text = example.get(key)
            if text:
                docs.append(text)
                count += 1
                if count % 10000 == 0:
                    print(f"  Collected {count}/{max_docs}")
    except Exception as e:
        print(f"Error loading {name}: {e}")

print(f"\nLoaded {len(docs)} docs in total")

## 7. Train/Val split

In [ ]:
random.seed(42)
random.shuffle(docs)

n_val = max(1, int(len(docs) * VAL_SPLIT))
val_docs = docs[:n_val]
train_docs = docs[n_val:]

print(f"Train: {len(train_docs)} docs")
print(f"Val:   {len(val_docs)} docs")

## 8. Process & save to Drive

In [ ]:
import json

# ── Train ──
print("\n→ Processing TRAIN set...")
t0 = time.time()
train_stats = build_dataset(train_docs, tokenizer, TOKENS_PER_SHARD, SHARD_DIR, desc="Train")
print(f"Train done in {time.time() - t0:.0f}s")

# ── Val ──
print("\n→ Processing VALIDATION set...")
t0 = time.time()
val_stats = build_dataset(val_docs, tokenizer, TOKENS_PER_SHARD, VAL_SHARD_DIR, desc="Val")
print(f"Val done in {time.time() - t0:.0f}s")

# ── Save metadata to Drive ──
meta = {
    "vocab_size": vocab_size,
    "train_shards": train_stats["shards"],
    "val_shards": val_stats["shards"],
    "total_train_tokens": train_stats["total_tokens"],
    "total_val_tokens": val_stats["total_tokens"],
    "tokens_per_shard": TOKENS_PER_SHARD,
    "chars_per_token": train_stats["chars_per_token"],
    "max_docs": MAX_DOCS,
    "shard_dir": SHARD_DIR,
    "val_shard_dir": VAL_SHARD_DIR,
}

with open(META_PATH, "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nMetadata saved to: {META_PATH}")
print(json.dumps(meta, indent=2))

---

✅ **Next:** `03_model.ipynb`

All outputs saved to **Google Drive**:
- `data_shards/` — training shards
- `data_shards_val/` — validation shards
- `dataset_meta.json` — metadata